# TP S2 : toucher la limite d'une machine

> **Salma :** « Le rapport mensuel du trafic fait planter le serveur. Pourquoi ? »

Ce notebook tourne dans un conteneur limité à **2 Go de mémoire**, comme un petit serveur. Vous allez :

1. mesurer ce qu'un fichier de clickstream coûte en mémoire ;
2. provoquer la saturation (volontairement) ;
3. contourner le problème en lisant par morceaux ;
4. extrapoler à 1 To et conclure.

Quatre checkpoints de 15 minutes. Notez vos réponses dans les cellules « Réponse ».

In [ ]:
import time
from pathlib import Path

import pandas as pd

DONNEES = Path("../../data")
FICHIER = DONNEES / "clickstream_aout.csv"

def limite_memoire():
    """Mémoire maximale accordée à ce conteneur (en octets), lue dans le cgroup."""
    for chemin in ["/sys/fs/cgroup/memory.max",
                   "/sys/fs/cgroup/memory/memory.limit_in_bytes"]:
        try:
            valeur = Path(chemin).read_text().strip()
            if valeur.isdigit() and int(valeur) < 2**60:
                return int(valeur)
        except OSError:
            pass
    return None

LIMITE = limite_memoire()
print("pandas", pd.__version__)
print("Limite mémoire du conteneur :", f"{LIMITE / 1e9:.1f} Go" if LIMITE else "aucune")

## Checkpoint 1 : un vrai fichier de clickstream (15 min)

Si le fichier n'est pas déjà copié depuis la clé USB, la cellule suivante le génère (1,5 Go, environ une minute). Le format est celui du jeu public REES46 : une ligne par événement (vue, ajout au panier, achat).

In [ ]:
if not FICHIER.exists():
    !python ../../scripts/generer_clickstream.py --taille 1.5G --sortie ../../data/clickstream_aout.csv
TAILLE = FICHIER.stat().st_size
print(f"Taille sur disque : {TAILLE / 1e9:.2f} Go")

In [ ]:
!head -n 4 ../../data/clickstream_aout.csv

**Réponse 1.** Combien de lignes environ ? (indice : une ligne fait à peu près 137 octets)

## Checkpoint 2 : combien de mémoire faudrait-il ? (15 min)

On charge seulement les 200 000 premières lignes, on mesure leur place sur disque et en mémoire, puis on extrapole.

In [ ]:
N = 200_000
with open(FICHIER, encoding="utf-8") as f:
    next(f)  # en-tête
    octets_disque = sum(len(next(f).encode("utf-8")) for _ in range(N))

echantillon = pd.read_csv(FICHIER, nrows=N)
octets_memoire = echantillon.memory_usage(deep=True).sum()
facteur = octets_memoire / octets_disque

print(f"{N:,} lignes : {octets_disque / 1e6:.0f} Mo sur disque, {octets_memoire / 1e6:.0f} Mo en mémoire")
print(f"Facteur disque -> mémoire : x{facteur:.2f}")
print(f"Pour tout le fichier : environ {facteur * TAILLE / 1e9:.1f} Go, "
      f"sans compter les copies temporaires pendant la lecture")
echantillon.dtypes

**Réponse 2.** Pourquoi une table prend-elle plus (ou autant) de place en mémoire que sur disque ? Regardez les types de colonnes.

Avant de lancer la cellule suivante :

1. **enregistrez le notebook** (Ctrl+S) ;
2. écrivez votre **prédiction** : que va-t-il se passer ?

In [ ]:
# Lecture naïve du fichier complet : le noyau risque de mourir, c'est voulu.
t0 = time.time()
df = pd.read_csv(FICHIER)
print(f"Chargé en {time.time() - t0:.0f} s : {len(df):,} lignes")

**Réponse 3.** Que s'est-il passé ? Si le message *Kernel restarting* ou *The kernel appears to have died* est apparu, le conteneur a dépassé sa limite et le système a tué le processus Python (*OOM killer*). **Relancez la première cellule** (imports) avant de continuer.

## Checkpoint 3 : lire par morceaux (15 min)

On ne garde en mémoire qu'un morceau de 500 000 lignes à la fois, on calcule un résultat partiel, puis on additionne les résultats partiels. Retenez ce schéma : c'est MapReduce en miniature (S4).

In [ ]:
TAILLE = FICHIER.stat().st_size
t0 = time.time()
comptes = pd.Series(dtype="int64")
nb_lignes = 0
for morceau in pd.read_csv(FICHIER, chunksize=500_000, usecols=["event_type"]):
    comptes = comptes.add(morceau["event_type"].value_counts(), fill_value=0)  # résultat partiel
    nb_lignes += len(morceau)
duree = time.time() - t0
DEBIT = TAILLE / duree

print(comptes.astype("int64").sort_values(ascending=False).to_string())
print(f"\n{nb_lignes:,} lignes en {duree:.0f} s, soit {DEBIT / 1e6:.0f} Mo/s")

In [ ]:
# Choisir ses colonnes et ses types change beaucoup la facture mémoire
N = 200_000
essais = {
    "toutes les colonnes": {},
    "3 colonnes utiles": dict(usecols=["event_type", "brand", "price"]),
    "3 colonnes + types compacts": dict(usecols=["event_type", "brand", "price"],
                                        dtype={"event_type": "category",
                                               "brand": "category",
                                               "price": "float32"}),
}
for nom, options in essais.items():
    octets = pd.read_csv(FICHIER, nrows=N, **options).memory_usage(deep=True).sum()
    print(f"{nom:<28} : {octets / 1e6:6.1f} Mo pour {N:,} lignes")

**Réponse 4.** Quelle mémoire utilise la lecture par morceaux ? Qu'a-t-on perdu par rapport à un DataFrame complet ?

## Checkpoint 4 : extrapoler (15 min)

Le débit mesuré ci-dessus est celui d'**une** machine. Que se passe-t-il quand les données grossissent ?

In [ ]:
volumes = [(TAILLE, "ce fichier"), (100e9, "100 Go"), (1e12, "1 To"), (10e12, "10 To")]
for volume, nom in volumes:
    secondes = volume / DEBIT
    print(f"{nom:>12} : {secondes / 3600:8.2f} h sur 1 machine | "
          f"{secondes / 100 / 60:8.2f} min sur 100 machines (idéal)")

**Réponse 5.** Salma veut chaque matin un rapport sur 3 ans d'historique (environ 18,5 To avec les journaux techniques, d'après le calcul de S1). Est-ce possible sur une seule machine ? Qu'est-ce qui limite : le processeur, la mémoire ou le disque ?

**Réponse 6.** Pourquoi 100 machines n'iront-elles pas exactement 100 fois plus vite ?

## À retenir

- Une table en mémoire coûte au moins sa taille sur disque, souvent plusieurs fois plus.
- Lire par morceaux contourne la limite de mémoire, pas la limite de temps.
- Pour aller plus vite, il faut **découper les données ET le calcul** sur plusieurs machines : c'est le sujet de S3 (HDFS) et S4 (MapReduce).